# Experiment 3: Bootstrap Stability Analysis (100 Resamples)

**Purpose:** Test whether SuStaIn-identified subtypes in structureless synthetic data are stable across bootstrap resamples.

**Expected result for null data:** ARI < 0.6 (artefactual subtypes should be unstable).

**Pre-specified criterion (Criterion 3):** ARI >= 0.6 across 100 bootstrap samples required for subtypes to be considered stable.

**Method:** Post-hoc subtype reassignment following Eshaghi et al. (2021). The full SuStaIn model is fitted once on the complete dataset; bootstrap samples are assigned to subtypes using learned model parameters rather than refitting.

---
*Colab setup: Runtime > Change runtime type > T4 GPU (optional, CPU works fine for bootstrap)*

In [ ]:
# Install dependencies
!pip install -q pySuStaIn numpy scipy scikit-learn pandas matplotlib seaborn tqdm

# Clone the repo for any custom utilities
!git clone --depth 1 https://github.com/Amelia3141/mphil.git mphil_repo 2>/dev/null || echo "Already cloned"
import sys
sys.path.insert(0, 'mphil_repo')
sys.path.insert(0, 'mphil_repo/pySuStaIn')

## 1. Synthetic Data Generation (No Encoded Subtypes)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm, multivariate_normal
from scipy.linalg import cholesky
import warnings
warnings.filterwarnings('ignore')

def generate_structureless_synthetic_data(n_patients=5000, seed=42):
    """
    Generate synthetic EDS/HSD data mimicking DICE Registry structure.
    CRITICALLY: No discrete subtypes are encoded. Any structure SuStaIn
    finds is emergent from correlations or algorithmic tendency to partition.
    """
    np.random.seed(seed)

    # 19 symptom variables across 7 clinical domains
    symptom_names = [
        'fatigue', 'chronic_pain', 'sleep_disturbance',           # Systemic
        'joint_pain', 'subluxations', 'joint_hypermobility',      # Musculoskeletal
        'gi_symptoms',                                             # Gastrointestinal
        'pots_dysautonomia',                                       # Autonomic
        'headaches', 'cognitive_dysfunction',                      # Neurological
        'anxiety', 'depression',                                   # Mental Health
        'mcas', 'allergies',                                       # Immune/Inflammatory
        'skin_fragility', 'urinary_symptoms', 'tmj_dysfunction',  # Other
        'vision_issues', 'hearing_tinnitus'                        # Other
    ]

    # Literature-based prevalence rates
    prevalences = {
        'fatigue': 0.78, 'chronic_pain': 0.82, 'sleep_disturbance': 0.65,
        'joint_pain': 0.85, 'subluxations': 0.70, 'joint_hypermobility': 0.98,
        'gi_symptoms': 0.75, 'pots_dysautonomia': 0.45,
        'headaches': 0.60, 'cognitive_dysfunction': 0.55,
        'anxiety': 0.65, 'depression': 0.45,
        'mcas': 0.25, 'allergies': 0.50,
        'skin_fragility': 0.70, 'urinary_symptoms': 0.40,
        'tmj_dysfunction': 0.35, 'vision_issues': 0.30,
        'hearing_tinnitus': 0.25
    }

    n_vars = len(symptom_names)

    # Correlation matrix based on clinical associations
    corr = np.eye(n_vars)
    name_to_idx = {name: i for i, name in enumerate(symptom_names)}

    def set_corr(a, b, r):
        i, j = name_to_idx[a], name_to_idx[b]
        corr[i, j] = r
        corr[j, i] = r

    # Strong correlations (r=0.45-0.60)
    set_corr('fatigue', 'chronic_pain', 0.55)
    set_corr('fatigue', 'sleep_disturbance', 0.50)
    set_corr('chronic_pain', 'sleep_disturbance', 0.45)
    set_corr('anxiety', 'depression', 0.55)
    set_corr('joint_pain', 'subluxations', 0.50)
    set_corr('gi_symptoms', 'mcas', 0.45)
    set_corr('pots_dysautonomia', 'fatigue', 0.45)
    set_corr('chronic_pain', 'depression', 0.45)

    # Moderate correlations (r=0.25-0.40)
    set_corr('anxiety', 'pots_dysautonomia', 0.35)
    set_corr('fatigue', 'cognitive_dysfunction', 0.35)
    set_corr('mcas', 'skin_fragility', 0.30)
    set_corr('mcas', 'allergies', 0.40)
    set_corr('headaches', 'cognitive_dysfunction', 0.35)
    set_corr('chronic_pain', 'anxiety', 0.30)
    set_corr('depression', 'fatigue', 0.35)
    set_corr('pots_dysautonomia', 'cognitive_dysfunction', 0.30)
    set_corr('joint_pain', 'joint_hypermobility', 0.35)
    set_corr('sleep_disturbance', 'cognitive_dysfunction', 0.30)

    # Weak cross-domain correlations (r=0.15-0.25)
    set_corr('fatigue', 'gi_symptoms', 0.20)
    set_corr('chronic_pain', 'headaches', 0.25)
    set_corr('anxiety', 'gi_symptoms', 0.20)
    set_corr('skin_fragility', 'subluxations', 0.15)
    set_corr('urinary_symptoms', 'gi_symptoms', 0.20)
    set_corr('tmj_dysfunction', 'headaches', 0.25)
    set_corr('vision_issues', 'headaches', 0.20)
    set_corr('hearing_tinnitus', 'tmj_dysfunction', 0.20)

    # Ensure positive semi-definite
    eigvals, eigvecs = np.linalg.eigh(corr)
    eigvals = np.maximum(eigvals, 1e-6)
    corr = eigvecs @ np.diag(eigvals) @ eigvecs.T
    np.fill_diagonal(corr, 1.0)

    # Generate latent continuous via Cholesky
    L = cholesky(corr, lower=True)
    z = np.random.randn(n_patients, n_vars)
    latent = z @ L.T

    # Threshold to ordinal (0=absent, 1=mild/occasional, 2=severe/constant)
    ordinal_data = np.zeros((n_patients, n_vars), dtype=int)
    for i, name in enumerate(symptom_names):
        prev = prevalences[name]
        threshold_absent = norm.ppf(1 - prev)
        # Among symptomatic: 60% mild, 40% severe
        threshold_severe = norm.ppf(1 - prev * 0.40)

        ordinal_data[:, i] = np.where(
            latent[:, i] < threshold_absent, 0,
            np.where(latent[:, i] < threshold_severe, 1, 2)
        )

    df = pd.DataFrame(ordinal_data, columns=symptom_names)

    # Add demographics
    df['age'] = np.clip(np.random.normal(36, 12, n_patients).astype(int), 18, 80)
    df['sex_female'] = np.random.binomial(1, 0.90, n_patients)
    diag_types = np.random.choice(
        ['hEDS', 'HSD', 'cEDS', 'vEDS', 'other'],
        size=n_patients, p=[0.75, 0.15, 0.07, 0.02, 0.01]
    )
    df['diagnosis_type'] = diag_types

    return df, symptom_names

# Generate data
df, symptom_names = generate_structureless_synthetic_data(n_patients=5000, seed=42)
print(f"Generated {len(df)} patients, {len(symptom_names)} symptoms")
print(f"\nPrevalence check (any symptom > 0):")
for s in symptom_names[:5]:
    print(f"  {s}: {(df[s] > 0).mean():.1%}")

## 2. Prepare Data for Ordinal SuStaIn

In [ ]:
# Prepare ordinal data for SuStaIn
# SuStaIn needs: the data matrix and probability arrays

from pySuStaIn.OrdinalSustain import OrdinalSustain
import matplotlib
matplotlib.use('Agg')

# Extract symptom data only
data = df[symptom_names].values.astype(float)
N_patients, N_biomarkers = data.shape

# Number of severity levels per biomarker (all have 3: 0, 1, 2)
N_S_gt = np.array([3] * N_biomarkers)

# SuStaIn biomarker labels
SuStaIn_labels = symptom_names

print(f"Data shape: {data.shape}")
print(f"Severity levels: {N_S_gt}")
print(f"Max possible events: {sum(N_S_gt - 1)} (biomarkers x (levels-1))")

## 3. Fit Baseline SuStaIn Model

In [ ]:
import os
import tempfile

# Create output directory
output_dir = tempfile.mkdtemp(prefix='sustain_bootstrap_')
print(f"Output directory: {output_dir}")

# Fit ordinal SuStaIn with k=2 subtypes (CVIC-optimal from null data test)
# Using modest settings for Colab runtime
sustain = OrdinalSustain(
    data,
    N_S_gt,
    N_biomarkers,
    SuStaIn_labels,
    N_startpoints=10,
    N_S_max=2,
    N_iterations_MCMC=5000,
    output_folder=output_dir,
    dataset_name='bootstrap_baseline',
    use_parallel_startpoints=False,
    seed=42
)

print("Fitting baseline SuStaIn model...")
samples_sequence, samples_f, ml_subtype, ml_stage, ml_likelihood = \
    sustain.run_sustain_algorithm(plot=False)

print(f"\nBaseline model fitted:")
print(f"  Subtypes found: {len(np.unique(ml_subtype))}")
for st in np.unique(ml_subtype):
    n = np.sum(ml_subtype == st)
    print(f"  Subtype {int(st)}: {n} patients ({n/len(ml_subtype):.1%})")

## 4. Run 100 Bootstrap Stability Resamples

In [ ]:
from sklearn.metrics import adjusted_rand_score
from tqdm import tqdm

def assign_subtypes_posthoc(sustain_model, data, samples_sequence, samples_f):
    """
    Post-hoc subtype assignment using learned model parameters.
    Following Eshaghi et al. (2021) approach: use the learned sequences
    and mixing fractions to assign new data points to subtypes.
    """
    N_S = samples_sequence.shape[0]  # number of subtypes
    n_samples_mcmc = samples_sequence.shape[2]
    N_patients = data.shape[0]

    # Use the most likely sequence for each subtype (mode of MCMC samples)
    # and compute likelihoods for each patient under each subtype
    from pySuStaIn.OrdinalSustain import OrdinalSustainData

    # Create sustainData for the new data
    prob_nl, prob_score = OrdinalSustain.static_prob_init(
        data, sustain_model.stage_biomarker_index,
        sustain_model.stage_score, sustain_model.prob_score
    )
    sustainData = OrdinalSustainData(prob_nl, prob_score, data.shape[0])

    # Get ML sequences
    ml_sequences = np.zeros((N_S, samples_sequence.shape[1]), dtype=int)
    for s in range(N_S):
        # Use mode of MCMC samples for each position
        from scipy import stats
        for pos in range(samples_sequence.shape[1]):
            ml_sequences[s, pos] = int(stats.mode(samples_sequence[s, pos, :], keepdims=False).mode)

    # Compute likelihood for each patient under each subtype
    total_prob_subtype = np.zeros((N_patients, N_S))
    for s in range(N_S):
        S = ml_sequences[s]
        p_perm_k = sustain_model._calculate_likelihood_stage(sustainData, S)
        # Sum across stages, weighted by uniform stage prior
        total_prob_subtype[:, s] = np.sum(p_perm_k, axis=1)

    # Weight by mixing fractions
    f_mean = np.mean(samples_f, axis=(1, 2)) if samples_f.ndim == 3 else np.mean(samples_f, axis=1)
    weighted = total_prob_subtype * f_mean[np.newaxis, :]

    # Assign to highest probability subtype
    assignments = np.argmax(weighted, axis=1)
    return assignments

# Run bootstrap stability analysis
n_bootstraps = 100
bootstrap_aris = []
bootstrap_assignments = []

print("Running 100 bootstrap resamples...")
for b in tqdm(range(n_bootstraps)):
    # Bootstrap resample (with replacement)
    rng = np.random.RandomState(seed=b + 1000)
    bootstrap_indices = rng.choice(len(data), size=len(data), replace=True)
    bootstrap_data = data[bootstrap_indices]

    # Assign bootstrap samples to subtypes using baseline model
    try:
        boot_assignments = assign_subtypes_posthoc(
            sustain, bootstrap_data, samples_sequence, samples_f
        )

        # Get original assignments for the same patients
        original_assignments = ml_subtype[bootstrap_indices]

        # Calculate ARI
        ari = adjusted_rand_score(original_assignments, boot_assignments)
        bootstrap_aris.append(ari)
        bootstrap_assignments.append(boot_assignments)
    except Exception as e:
        print(f"Bootstrap {b} failed: {e}")
        bootstrap_aris.append(np.nan)

bootstrap_aris = np.array(bootstrap_aris)
valid_aris = bootstrap_aris[~np.isnan(bootstrap_aris)]
print(f"\nCompleted {len(valid_aris)}/{n_bootstraps} bootstraps")
print(f"Mean ARI: {np.nanmean(bootstrap_aris):.3f}")
print(f"Median ARI: {np.nanmedian(bootstrap_aris):.3f}")
print(f"Std ARI: {np.nanstd(bootstrap_aris):.3f}")
print(f"Range: [{np.nanmin(bootstrap_aris):.3f}, {np.nanmax(bootstrap_aris):.3f}]")

## 5. Evaluate Against Pre-Specified Criterion 3

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Criterion 3: Bootstrap stability requires ARI >= 0.6
THRESHOLD = 0.6

mean_ari = np.nanmean(bootstrap_aris)
pct_above = np.nanmean(bootstrap_aris >= THRESHOLD) * 100

print("=" * 60)
print("CRITERION 3: BOOTSTRAP STABILITY")
print("=" * 60)
print(f"Threshold: ARI >= {THRESHOLD}")
print(f"Mean ARI: {mean_ari:.3f}")
print(f"Bootstraps above threshold: {pct_above:.1f}%")
print(f"Result: {'PASS' if mean_ari >= THRESHOLD else 'FAIL'}")
print("=" * 60)

if mean_ari < THRESHOLD:
    print("\nInterpretation: Subtypes are UNSTABLE across bootstrap resamples.")
    print("This is expected for artefactual subtypes in structureless data.")
    print("The validation framework correctly identifies these as unreliable.")
else:
    print("\nWARNING: Unexpectedly high stability for structureless data.")
    print("This warrants investigation, may indicate correlation-driven structure.")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
axes[0].hist(valid_aris, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold (ARI={THRESHOLD})')
axes[0].axvline(mean_ari, color='orange', linestyle='-', linewidth=2, label=f'Mean ARI={mean_ari:.3f}')
axes[0].set_xlabel('Adjusted Rand Index')
axes[0].set_ylabel('Count')
axes[0].set_title('Bootstrap Stability: ARI Distribution')
axes[0].legend()

# Box plot
axes[1].boxplot(valid_aris, vert=True)
axes[1].axhline(THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold={THRESHOLD}')
axes[1].set_ylabel('Adjusted Rand Index')
axes[1].set_title('Bootstrap ARI')
axes[1].legend()

plt.tight_layout()
plt.savefig('bootstrap_stability_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSaved: bootstrap_stability_results.png")

## 6. Model Selection Robustness (Criterion 4)

In [ ]:
# Criterion 4: Optimal k should be consistent across bootstraps
# Run CVIC for k=1,2,3 on a subset of bootstraps (computationally expensive)

n_robustness_bootstraps = 20  # Subset for tractability
optimal_k_per_bootstrap = []

print(f"Testing model selection robustness on {n_robustness_bootstraps} bootstraps...")
for b in tqdm(range(n_robustness_bootstraps)):
    rng = np.random.RandomState(seed=b + 2000)
    boot_idx = rng.choice(len(data), size=len(data), replace=True)
    boot_data = data[boot_idx]

    boot_output = tempfile.mkdtemp(prefix=f'sustain_robustness_{b}_')

    try:
        boot_sustain = OrdinalSustain(
            boot_data,
            N_S_gt,
            N_biomarkers,
            SuStaIn_labels,
            N_startpoints=5,
            N_S_max=3,
            N_iterations_MCMC=2000,
            output_folder=boot_output,
            dataset_name=f'robustness_{b}',
            use_parallel_startpoints=False,
            seed=b + 2000
        )

        # Use cross-validation to select optimal k
        CVIC, loglike = boot_sustain.cross_validate_sustain_model(test_idxs=None, select_fold=None)
        optimal_k = np.argmin(CVIC) + 1
        optimal_k_per_bootstrap.append(optimal_k)
    except Exception as e:
        print(f"Robustness bootstrap {b} failed: {e}")

optimal_k_arr = np.array(optimal_k_per_bootstrap)
modal_k = int(np.median(optimal_k_arr))
pct_consistent = np.mean(np.abs(optimal_k_arr - modal_k) <= 1) * 100

print(f"\n{'='*60}")
print("CRITERION 4: MODEL SELECTION ROBUSTNESS")
print(f"{'='*60}")
print(f"Modal k: {modal_k}")
print(f"Distribution: {dict(zip(*np.unique(optimal_k_arr, return_counts=True)))}")
print(f"Consistent (within +/-1 of mode): {pct_consistent:.1f}%")
print(f"Threshold: >= 70%")
print(f"Result: {'PASS' if pct_consistent >= 70 else 'FAIL'}")
print(f"{'='*60}")

## 7. Summary of Results

In [ ]:
# Compile all results
print("=" * 70)
print("VALIDATION RESULTS SUMMARY - STRUCTURELESS SYNTHETIC DATA")
print("=" * 70)
print(f"{'Criterion':<45} {'Result':<8} {'Value'}")
print("-" * 70)
print(f"{'1. Prevalence >= 10%':<45} {'PASS':<8} Min 48.0%")
print(f"{'2. Distinctiveness (d>=0.5 on >=3 domains)':<45} {'FAIL':<8} Max d=0.27")
print(f"{'3. Bootstrap stability (ARI >= 0.6)':<45} {'PASS' if mean_ari >= 0.6 else 'FAIL':<8} ARI={mean_ari:.3f}")
print(f"{'4. Model selection robustness (70%)':<45} {'PASS' if pct_consistent >= 70 else 'FAIL':<8} {pct_consistent:.1f}%")
print(f"{'5. Not severity-only (r < 0.5)':<45} {'PASS':<8} r=0.047")
print(f"{'6. Plausible sequences':<45} {'N/A':<8} Moot (Crit 2 failed)")
print("-" * 70)

# Save results
results = {
    'bootstrap_aris': bootstrap_aris.tolist(),
    'mean_ari': float(mean_ari),
    'median_ari': float(np.nanmedian(bootstrap_aris)),
    'std_ari': float(np.nanstd(bootstrap_aris)),
    'criterion_3_pass': bool(mean_ari >= 0.6),
    'optimal_k_distribution': optimal_k_per_bootstrap,
    'criterion_4_pct_consistent': float(pct_consistent),
    'criterion_4_pass': bool(pct_consistent >= 70),
}

import json
with open('bootstrap_stability_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved: bootstrap_stability_results.json")